In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.window import Window as w
from pyspark.sql.functions import rank,col, trim, lower, to_date, current_date, regexp_replace

In [0]:
def read_data(table_name) -> DataFrame:
    """
    Reads data from the bronze table name provided as input

    args:
        table_name: name of the table
    """
    return spark.read.table(table_name)

In [0]:
def save_table(df, table_name):
    """
    Saves the DataFrame to the silver table name provided as input

    args:
        df: Spark DataFrame
        table_name: name of the table
    """
    df.write.mode("overwrite").saveAsTable(f"PROJECT1001.SILVER.{table_name}_silver")


In [0]:

def handle_customers_missing_values(df: DataFrame) -> DataFrame:

    """
    Returns a DataFrame with missing values handled for the columns name, city, state, country, is_active

    args:
        df: Customers DataFrame
    """
    return df.fillna({"name":"UNKNOWN", "city": "UNKNOWN", "state": "UNKNOWN", "country": "UNKNOWN", "is_active": False})

def handle_items_missing_values(df: DataFrame) -> DataFrame:

    """
    Returns a DataFrame with missing values handled for the columns order_id, item_name, category, price

    args:
        df: Items DataFrame
    """
    return df.fillna({"order_id":0,"item_name":"UNKNOWN", "category": "UNKNOWN","price":0})

def handle_orders_missing_values(df: DataFrame) -> DataFrame:

    """
    Returns a DataFrame with missing values handled for the columns customer_id, status, total_amount

    args:
        df: Orders DataFrame
    """
    return df.fillna({"customer_id":0,"status":"UNKNOWN", "total_amount":0})

def handle_payments_missing_values(df: DataFrame) -> DataFrame:

    """
    Returns a DataFrame with missing values handled for the columns order_id, amount, payment_method

    args:
        df: Payments DataFrame
    """
    return df.fillna({"order_id":0,"amount":0, "payment_method": "UNKNOWN"})

def handle_shippings_missing_values(df: DataFrame) -> DataFrame:

    """
    Returns a DataFrame with missing values handled for the columns order_id, shipping_address, shipping_method

    args:
        df: Shippings DataFrame
    """
    return df.fillna({"order_id":0,"shipping_address":"UNKNOWN", "shipping_method": "UNKNOWN"})

In [0]:

def remove_duplicate_customers(df : DataFrame) -> DataFrame:
    """
    Returns customer dataframe after removing duplicate rows based on customer_id, keeping latest records

    args:
        df: Customers DataFrame
    """
    ws= w.partitionBy("customer_id").orderBy(col("registration_date").desc())
    return df.withColumn("rank", rank().over(ws)).filter("rank=1").drop("rank")

def remove_duplicate_items(df: DataFrame) -> DataFrame:
    """
    Returns items dataframe after removing duplicate rows based on item_id

    args:
        df: Items DataFrame
    """
    return df.dropDuplicates(subset=['item_id'])

def remove_duplicate_orders(df: DataFrame) -> DataFrame:
    """
    Returns orders dataframe after removing duplicate rows based on order_id, keeping latest records
    
    args:
        df: Orders DataFrame
    """
    ws= w.partitionBy("order_id").orderBy(col("order_date").desc())
    return df.withColumn("rank", rank().over(ws)).filter("rank=1").drop("rank")

def remove_duplicate_payments(df: DataFrame) -> DataFrame:
    """
    Returns payments dataframe after removing duplicate rows based on payment_id, keeping latest records

    args:
        df: Payments DataFrame
    """
    ws= w.partitionBy("payment_id").orderBy(col("payment_date").desc())
    return df.withColumn("rank", rank().over(ws)).filter("rank=1").drop("rank")

def remove_duplicate_shippings(df: DataFrame) -> DataFrame:
    """
    Returns shippings dataframe after removing duplicate rows based on shipping_id, keeping latest records

    args:
        df: Shippings DataFrame
    """
    ws= w.partitionBy("shipping_id").orderBy(col("shipping_date").desc())
    return df.withColumn("rank", rank().over(ws)).filter("rank=1").drop("rank")

In [0]:
def format_string_columns(df: DataFrame, columns: list) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Format string columns to lowercase and trim and replace spaces with underscores
    
    args:
        df: Spark DataFrame
        columns: list of string columns
    """
    for column in columns:
        df = df.withColumn(column, lower(regexp_replace(trim(col(column))," ", "_")))
    return df


In [0]:
def format_date_columns(df: DataFrame, columns: list) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Format date columns to yyyy-MM-dd
    
    args:
        df: Spark DataFrame
        columns: list of datetype columns to format
    """
    for column in columns:
        df = df.withColumn(column, to_date(col(column), "yyyy-MM-dd"))
    return df

def format_numeric_columns(df: DataFrame, columns: list) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Format numeric columns to int
    
    args:
        df: Spark DataFrame
        columns: list of numeric columns to format
    """
    for column in columns:
        df = df.withColumn(column, col(column).cast("int"))
    return df
def format_decimal_columns(df, columns):
    for column in columns:
        df = df.withColumn(column, col(column).cast("decimal(10,2)"))
    return df


In [0]:
def transform_customers(df: DataFrame) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Missing values handled for the columns name, city, state, country, is_active
        - Remove duplicate rows based on customer_id, keeping latest records
        - Format string columns to lowercase and trim
        - Format date columns to yyyy-MM-dd
        - Format customer_id to int
        - Ensure no unexpected future dates

    args:
        df: Customers DataFrame
    """""
    df = handle_customers_missing_values(df)
    df = remove_duplicate_customers(df)
    df = format_string_columns(df, ["name", "city", "state", "country"])
    df = format_date_columns(df, ["registration_date"])
    df = format_numeric_columns(df, ["customer_id"])
    df = df.filter(df.registration_date <= current_date())
    return df

In [0]:
def transform_items(df: DataFrame) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Missing values handled for the columns order_id, item_name, category, price
        - Remove duplicate rows based on item_id
        - Format string columns to lowercase and trim
        - Format item_id, order_id to int
        - Format price to decimal(10,2)
        - Ensure no unexpected future dates
        - Remove Impossible Values, i.e. no unexpected negative values for numeric columns

    args:
        df: Items DataFrame
    """""
    df = handle_items_missing_values(df)
    df = remove_duplicate_items(df)
    df = format_string_columns(df, ["item_name", "category"])
    df = format_numeric_columns(df, ["item_id", "order_id", "price"])
    df = format_decimal_columns(df, ["price"])
    df = df.filter(df.price>0)

    return df

In [0]:
def transform_orders(df: DataFrame) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Missing values handled for the columns order_id, customer_id, status, total_amount
        - Remove duplicate rows based on order_id, keeping latest records
        - Format string columns to lowercase and trim
        - Format date columns to yyyy-MM-dd
        - Format order_id, customer_id to int
        - Format total_amount to decimal(10,2)
        - Ensure no unexpected future dates
        - Remove Impossible Values, i.e. no unexpected negative values for numeric columns

    args:
        df: Orders DataFrame
    """""
    df = handle_orders_missing_values(df)
    df = remove_duplicate_orders(df)
    df = format_string_columns(df, ["status"])
    df = format_date_columns(df, ["order_date"])
    df = format_numeric_columns(df, ["order_id", "customer_id"])
    df = format_decimal_columns(df, ["total_amount"])
    df = df.filter((df.order_date <= current_date()) & (df.total_amount>=0))

    return df

In [0]:
def transform_payments(df: DataFrame) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Missing values handled for the columns order_id, amount, payment_method
        - Remove duplicate rows based on payment_id, keeping latest records
        - Format string columns to lowercase and trim
        - Format date columns to yyyy-MM-dd
        - Format payment_id, order_id to int
        - Format amount to decimal(10,2)
        - Ensure no unexpected future dates
        - Remove Impossible Values, i.e. no unexpected negative values for numeric columns

    args:
        df: Payments DataFrame
    """""
    df = handle_payments_missing_values(df)
    df = remove_duplicate_payments(df)
    df = format_string_columns(df, ["payment_method"])
    df = format_date_columns(df, ["payment_date"])
    df = format_numeric_columns(df, ["payment_id", "order_id"])
    df = format_decimal_columns(df, ["amount"])
    df = df.filter((df.payment_date <= current_date()) & (df.amount>=0))

    return df

In [0]:
def transform_shippings(df: DataFrame) -> DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Missing values handled for the columns order_id, shipping_date, shipping_cost
        - Remove duplicate rows based on shipping_id, keeping latest records
        - Format string columns to lowercase and trim
        - Format date columns to yyyy-MM-dd
        - Format shipping_id, order_id to int
        - Ensure no unexpected future dates
        - Remove Impossible Values, i.e. no unexpected negative values for numeric columns

    args:
        df: Shippings DataFrame
    """
    df = handle_shippings_missing_values(df)
    df = remove_duplicate_shippings(df)
    df = format_string_columns(df, ["shipping_method", "shipping_address"])
    df = format_date_columns(df, ["shipping_date"])
    df = format_numeric_columns(df, ["shipping_id", "order_id"])
    df = df.filter(df.shipping_date <= current_date())

    return df

In [0]:
def build_order_details(orders_df: DataFrame, customers_df: DataFrame, items_df: DataFrame, payments_df: DataFrame, shippings_df: DataFrame) ->DataFrame:
    """
    Returns a Spark DataFrame with the following transformations applied:
        - Join the order_df with customers_df, items_df, payments_df, shippings_df
    
    args:
        orders_df: Orders DataFrame
        customers_df: Customers DataFrame
        items_df: Items DataFrame
        payments_df: Payments DataFrame
        shippings_df: Shippings DataFrame
    """   
    o = orders_df.alias("o")
    c = customers_df.alias("c")
    i = items_df.alias("i")
    p = payments_df.alias("p")
    s = shippings_df.alias("s")

    df = (
            o.join(c, o.customer_id == c.customer_id, "left")
            .join(i, o.order_id == i.order_id, "left")
            .join(p, o.order_id == p.order_id, "left")
            .join(s, o.order_id == s.order_id, "left")
        )\
            .select("o.order_id","o.customer_id","o.order_date","o.total_amount","o.status",
                    "c.name","c.city","c.state","c.country","c.registration_date","c.is_active",
                    "i.item_id","i.item_name","i.category","i.price",
                    "p.payment_id","p.payment_date","p.amount","p.payment_method",
                    "s.shipping_id", "s.shipping_date", "s.shipping_address", "s.shipping_method"
                    )
    return df